# Tool Calling for sLLM
- test sLLM can use tools
- test tool modularized, NOT function
- Reference: Prompt template supporting tool calling for sLLM in instruction-tuning
  - https://ollama.com/cow/gemma2_tools

In [1]:
embedding_model_name = "gemma-2-embed"
lang_model_name = "tiger-gemma2"

max_tokens = 1024

In [2]:
import uuid

from abc import abstractmethod
from pprint import pprint
from pydantic import BaseModel, Field
from typing import Annotated, Callable, List, Literal, Optional
from typing_extensions import TypedDict

from langchain.schema import AIMessage, HumanMessage, SystemMessage

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import AnyMessage, add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from langchain_community.embeddings import OllamaEmbeddings

from langchain_core.messages import ToolMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import Runnable, RunnableConfig, RunnableLambda
from langchain_core.tools import tool
from langchain_core.tools.base import BaseTool

from langchain_ollama import ChatOllama

In [3]:
llm = ChatOllama(
    model=lang_model_name,
    temperature=0.8,
    num_predict=max_tokens,
    num_gpu=-1,
    # config={"extra": "allow"}
    # extra_fields_behavior="allow",
)

In [4]:
llm = llm.with_fallbacks([llm])

In [5]:
class AddNumbersTool(BaseTool):
    name: str = "add_numbers"
    description: str = """
    두 숫자가 주어졌을 때 숫자의 합을 계산하고 반환한다.
    """
    def _run(self, a, b):
        return str(a + b)

    async def _arun(self, a, b):
        return str(a + b)
    
    def parse_and_run(self, tool_action):
        args = [int(x.strip()) for x in tool_action["action_input"].split(",")]
        result = self._run(args[0], args[1])
        action_output = {
            "action": "add_numbers",
            "output": result
        }
        return action_output

add_tool = AddNumbersTool()

In [39]:
class SubtractNumbersTool(BaseTool):
    name: str = "subtract_numbers"
    description: str = """
    두 숫자가 주어졌을 때 숫자의 차를 계산하고 반환한다.
    """
    def _run(self, a, b):
        return str(a - b)

    async def _arun(self, a, b):
        return str(a - b)

    def parse_and_run(self, tool_action):
        args = [int(x.strip()) for x in tool_action["action_input"].split(",")]
        result = self._run(args[0], args[1])
        action_output = {
            "action": "subtract_numbers",
            "output": result
        }
        return action_output

subtract_tool = SubtractNumbersTool()

In [121]:
import json

from langchain import hub
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

prompt_for_extract_actions = hub.pull("kwonempty/extract-actions-for-ollama")

prompt_prev_tool_output = '8. 두번째 도구부터 action_input의 첫번째 인자는 \'PREV_TOOL_OUTPUT\'으로 지정하고 인자 값 구분자는 콤마(,)로 한다.'
prompt_texts = prompt_for_extract_actions.messages[0].prompt.template.split('\n')
prompt_text_with_prev_tool_output = '\n'.join(prompt_texts[:29] + [prompt_prev_tool_output] + prompt_texts[-5:])
prompt_for_extract_actions.messages[0].prompt.template = prompt_text_with_prev_tool_output

print(prompt_for_extract_actions.messages[0].prompt.template)

/home/ed/miniconda3/envs/mypy311/lib/python3.11/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(



당신은 인간의 질문에 답변하기 위해 적절한 도구를 선택하는 AI 어시스턴트입니다. 

다음 도구들을 사용할 수 있습니다:
{tools}

인간의 질문을 주의 깊게 분석하고, 가장 적절한 도구를 선택하여 답변하세요. 질문에 따라 여러 도구를 사용해야 할 수도 있습니다.

응답 시 다음 JSON 형식을 엄격히 따라주세요:
```json
[
  {{
    "action": string, // 선택한 도구의 이름 (tool_name)
    "action_input": string // 도구에 입력할 검색어 또는 질문
  }},
  {{
    // 다음 액션 정보
  }}
]
```

응답 지침:
1. 항상 JSON 배열로 응답하세요, 단일 도구를 사용하는 경우에도 마찬가지입니다.
2. 하나의 도구만 필요한 경우, 배열에 하나의 객체만 포함시키세요.
3. 여러 도구가 필요한 경우, 각 도구에 대해 별도의 객체를 배열에 추가하세요.
4. 액션의 순서가 중요한 경우, 배열 내 객체의 순서로 표현하세요.
5. 이 JSON 형식으로만 응답하고, 다른 설명이나 추가 텍스트는 포함하지 마세요.
6. 인간의 질문에 직접 답변하지 말고, 적절한 도구를 선택하여 JSON 형식으로만 응답하세요.
7. 적절한 도구를 찾지 못하거나 도구 사용이 필요하지 않다고 판단되는 경우, "action"을 "None"으로, "action_input"을 빈 문자열로 설정하여 응답하세요.
8. 두번째 도구부터 action_input의 첫번째 인자는 'PREV_TOOL_OUTPUT'으로 지정하고 인자 값 구분자는 콤마(,)로 한다.

question: {question}

answer: 



In [122]:
TOOLS_KEY = "tool_actions"
TOOL_NAME_KEY = "action"
TOOL_DESCRIPTION = "description"

tools = [add_tool, subtract_tool]
TOOL_NAMES = [t.name for t in tools]
TOOL_MAPPING = {t.name: t for t in tools}

def list_tools(query) -> str:
    """
    사용 가능한 도구들의 이름과 설명을 JSON 문자열 형식으로 변환하여 반환
    """
    # tools 리스트에서 각 도구의 이름, 설명을 딕셔너리 형태로 추출
    tool_info = [{TOOL_NAME_KEY: tool.name, TOOL_DESCRIPTION: tool.description} for tool in tools]

    # print("[list_tools/tool_info]: {tool_info}")

    return {
        TOOLS_KEY: json.dumps(
            tool_info,
            ensure_ascii=False
        ),
        "question": query
    }

def execute_tool_action(tool: dict, tool_output_history: list) -> dict:
    """
    도구 액션을 실행하고 결과를 반환
    """
    print(f"#### [EXECUTE-TOOL-ACTION] {tool}")

    if tool_output_history:
        inputs = tool["action_input"].split(',')
        inputs[0] = tool_output_history[-1]["output"]
        tool["action_input"] = ','.join(inputs)

    action_output = TOOL_MAPPING[tool[TOOL_NAME_KEY]].parse_and_run(tool)
    print(f"#### [ACTION-OUTPUT] {action_output}")
    return action_output
    
def parse_answer_for_actions(action_output: str):
    print(f"#### {action_output.__dict__}")
    return json.loads(action_output)

def run_through_actions(tool_actions: list):
    results = []
    print(f"#### [RUNNING-WITH-ACTIONS] {tool_actions}")

    for index, tool in enumerate(tool_actions):
        print(f"#### [RUNNING-WITH-TOOL-{index}] {TOOL_NAME_KEY}: {tool}")

        # 도구 액션을 실행하고 결과를 저장
        action_output = execute_tool_action(tool, results)
        results.append(action_output)
        # logging action event
        print(f"#### {TOOL_NAME_KEY}: {tool} => {results}")

    return results


chain_for_extract_actions = (
    {
        "tools": list_tools,
        "question": RunnablePassthrough()
    }
    | prompt_for_extract_actions 
    | llm
    | JsonOutputParser()
    | RunnableLambda(run_through_actions)
    # | RunnableLambda(parse_answer_for_actions)
    # | RunnablePassthrough.assign(
    #     tool_actions=lambda x: x["tool_actions"],
    #     question=lambda x: x["question"]
    # )
    # | StrOutputParser()
)

In [ ]:
def retry_invoke(invoke_func, n_try=3, **kwargs):
    attempts = 0
    while n_try > attempts:
        try:
            invoke_func(kwargs)
        except Exception as e:
            # TODO logging chain invoking event
            print(f"#### [RETRY_INVOKE][{type(e)}]: {str(e)}")
            attempts += 1
            if attempts == n_try:
                raise Exception(f"{n_try} trials for {kwargs}")
        else:
            break

In [115]:
question = "Please add 8 and 7 using the tool. Just answer in short."
action_outputs: list[dict[str, str]] = chain_for_extract_actions.invoke(question)
action_outputs

#### [RUNNING-WITH-ACTIONS] [{'action': 'add_numbers', 'action_input': '8,7'}]
#### [RUNNING-WITH-TOOL-0] action: {'action': 'add_numbers', 'action_input': '8,7'}
#### [EXECUTE-TOOL-ACTION] {'action': 'add_numbers', 'action_input': '8,7'}
#### [ACTION-OUTPUT] {'action': 'add_numbers', 'output': '15'}
#### action: {'action': 'add_numbers', 'action_input': '8,7'} => [{'action': 'add_numbers', 'output': '15'}]


[{'action': 'add_numbers', 'output': '15'}]

In [49]:
question = "Please subtract 8 and 7 using the tool. Just answer in short."
action_outputs: list[dict[str, str]] = chain_for_extract_actions.invoke(question)
action_outputs

[list_tools/tool_info]: [{'action': 'add_numbers', 'description': '\n    두 숫자가 주어졌을 때 숫자의 합을 계산하고 반환한다.\n    '}, {'action': 'subtract_numbers', 'description': '\n    두 숫자가 주어졌을 때 숫자의 차를 계산하고 반환한다.\n    '}]
#### [RUNNING] [{'action': 'subtract_numbers', 'action_input': '8,7'}]
#### [RUNNING-WITH-TOOL-0] action: {'action': 'subtract_numbers', 'action_input': '8,7'}
#### [EXECUTE-TOOL-ACTION] {'action': 'subtract_numbers', 'action_input': '8,7'}
#### [ACTION-OUTPUT] {'action': 'subtract_numbers', 'output': '1'}
#### action: {'action': 'subtract_numbers', 'action_input': '8,7'} => [{'action': 'subtract_numbers', 'output': '1'}]


[{'action': 'subtract_numbers', 'output': '1'}]

In [123]:
question = "Please add 8 and 7, and subtract its result and 5 using the tool. Just answer in short."
action_outputs: list[dict[str, str]] = chain_for_extract_actions.invoke(question)
action_outputs

#### [RUNNING-WITH-ACTIONS] [{'action': 'add_numbers', 'action_input': '8,7'}, {'action': 'subtract_numbers', 'action_input': 'PREV_TOOL_OUTPUT,5'}]
#### [RUNNING-WITH-TOOL-0] action: {'action': 'add_numbers', 'action_input': '8,7'}
#### [EXECUTE-TOOL-ACTION] {'action': 'add_numbers', 'action_input': '8,7'}
#### [ACTION-OUTPUT] {'action': 'add_numbers', 'output': '15'}
#### action: {'action': 'add_numbers', 'action_input': '8,7'} => [{'action': 'add_numbers', 'output': '15'}]
#### [RUNNING-WITH-TOOL-1] action: {'action': 'subtract_numbers', 'action_input': 'PREV_TOOL_OUTPUT,5'}
#### [EXECUTE-TOOL-ACTION] {'action': 'subtract_numbers', 'action_input': 'PREV_TOOL_OUTPUT,5'}
#### [ACTION-OUTPUT] {'action': 'subtract_numbers', 'output': '10'}
#### action: {'action': 'subtract_numbers', 'action_input': '15,5'} => [{'action': 'add_numbers', 'output': '15'}, {'action': 'subtract_numbers', 'output': '10'}]


[{'action': 'add_numbers', 'output': '15'},
 {'action': 'subtract_numbers', 'output': '10'}]

In [35]:

retry_invoke(chain_for_extract_actions.invoke, n_try=3, question=question)
# try:
#     chain_for_extract_actions.invoke(question)
# except Exception as jde:
#     print(f"#### [{type(jde)}]: {str(jde)}")
#     chain_for_extract_actions.invoke(question)

[get_tools/tool_info]: [{'action': 'add_numbers', 'description': '\n    두 숫자가 주어졌을 때 숫자의 합을 계산하고 반환한다.\n    '}]
#### [{'action': 'add_numbers', 'action_input': '8,7'}]
#### action: {'action': 'add_numbers', 'action_input': '8,7'}
#### action: {'action': 'add_numbers', 'action_input': '8,7'} => [{'action': 'add_numbers', 'output': '15'}]


In [38]:
type(prompt_for_extract_actions)

langchain_core.prompts.chat.ChatPromptTemplate

In [10]:
prompt_for_extract_actions.__dict__

{'name': None,
 'input_variables': ['question', 'tools'],
 'optional_variables': [],
 'input_types': {},
 'output_parser': None,
 'partial_variables': {},
 'metadata': {'lc_hub_owner': 'kwonempty',
  'lc_hub_repo': 'extract-actions-for-ollama',
  'lc_hub_commit_hash': '0a6e97369e4f5af0ed20196b40480c7178e171f92fa0e0c7599a5e7809ee4ad3'},
 'tags': None,
 'messages': [SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question', 'tools'], input_types={}, partial_variables={}, template='\n당신은 인간의 질문에 답변하기 위해 적절한 도구를 선택하는 AI 어시스턴트입니다. \n\n다음 도구들을 사용할 수 있습니다:\n{tools}\n\n인간의 질문을 주의 깊게 분석하고, 가장 적절한 도구를 선택하여 답변하세요. 질문에 따라 여러 도구를 사용해야 할 수도 있습니다.\n\n응답 시 다음 JSON 형식을 엄격히 따라주세요:\n```json\n[\n  {{\n    "action": string, // 선택한 도구의 이름 (tool_name)\n    "action_input": string // 도구에 입력할 검색어 또는 질문\n  }},\n  {{\n    // 다음 액션 정보\n  }}\n]\n```\n\n응답 지침:\n1. 항상 JSON 배열로 응답하세요, 단일 도구를 사용하는 경우에도 마찬가지입니다.\n2. 하나의 도구만 필요한 경우, 배열에 하나의 객체만 포함시키세요.\n3. 여러 도구가 필요한 경우, 각 도구에 대해 별도의 객체를 배열에 추가하세요.\n4

In [11]:
len(prompt_for_extract_actions.messages[-1].prompt.template)

742

In [12]:
pprint(prompt_for_extract_actions.messages[-1].prompt.template)

('\n'
 '당신은 인간의 질문에 답변하기 위해 적절한 도구를 선택하는 AI 어시스턴트입니다. \n'
 '\n'
 '다음 도구들을 사용할 수 있습니다:\n'
 '{tools}\n'
 '\n'
 '인간의 질문을 주의 깊게 분석하고, 가장 적절한 도구를 선택하여 답변하세요. 질문에 따라 여러 도구를 사용해야 할 수도 있습니다.\n'
 '\n'
 '응답 시 다음 JSON 형식을 엄격히 따라주세요:\n'
 '```json\n'
 '[\n'
 '  {{\n'
 '    "action": string, // 선택한 도구의 이름 (tool_name)\n'
 '    "action_input": string // 도구에 입력할 검색어 또는 질문\n'
 '  }},\n'
 '  {{\n'
 '    // 다음 액션 정보\n'
 '  }}\n'
 ']\n'
 '```\n'
 '\n'
 '응답 지침:\n'
 '1. 항상 JSON 배열로 응답하세요, 단일 도구를 사용하는 경우에도 마찬가지입니다.\n'
 '2. 하나의 도구만 필요한 경우, 배열에 하나의 객체만 포함시키세요.\n'
 '3. 여러 도구가 필요한 경우, 각 도구에 대해 별도의 객체를 배열에 추가하세요.\n'
 '4. 액션의 순서가 중요한 경우, 배열 내 객체의 순서로 표현하세요.\n'
 '5. 이 JSON 형식으로만 응답하고, 다른 설명이나 추가 텍스트는 포함하지 마세요.\n'
 '6. 인간의 질문에 직접 답변하지 말고, 적절한 도구를 선택하여 JSON 형식으로만 응답하세요.\n'
 '7. 적절한 도구를 찾지 못하거나 도구 사용이 필요하지 않다고 판단되는 경우, "action"을 "None"으로, '
 '"action_input"을 빈 문자열로 설정하여 응답하세요.\n'
 '\n'
 'question: {question}\n'
 '\n'
 'answer: \n')


In [13]:
chain_for_execution = Runnable

In [14]:
llm_chain = RunnablePassthrough() | llm | StrOutputParser()

In [15]:
prompt = prompt_for_extract_actions

In [16]:
# from langchain.agents import AgentExecutor, create_tool_calling_agent, tool

# agent = create_tool_calling_agent(llm_chain, tools, prompt)
# agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# agent_executor.invoke({"input": "what is the value of magic_function(3)?"})


In [17]:
from langchain.agents import AgentExecutor, create_tool_calling_agent, tool
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant"),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)
# model = ChatOpenAI(model="gpt-4o-mini")
model = llm

@tool
def magic_function(input: int) -> int:
    """Applies a magic function to an input."""
    return input + 2

tools = [magic_function]

agent = create_tool_calling_agent(model, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_executor.invoke({"input": "what is the value of magic_function(3)?"})

# Using with chat history
from langchain_core.messages import AIMessage, HumanMessage
agent_executor.invoke(
    {
        "input": "what's my name?",
        "chat_history": [
            HumanMessage(content="hi! my name is bob"),
            AIMessage(content="Hello Bob! How can I assist you today?"),
        ],
    }
)



> Entering new AgentExecutor chain...


ResponseError: tiger-gemma2 does not support tools

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a helpful assistant.
            If a user requests that {add_tool.description}, delegate the task to the appropriate specialized assistant by invoking the corresponding tool.
            """.strip(),
        ),
        # (
        #     "placehoder",
        #     "{messages}"
        # )
    ]
)

# 사용자 메시지 생성
user_message = HumanMessage(content="Please add 8 and 7 using the tool. Just answer in short.")

runnable = prompt | llm.bind_tools([
    add_tool
])

response = runnable.invoke("Please add 8 and 7 using the tool. Just answer in short.")

pprint(response)

# LLM 호출 및 도구 사용 로직 처리
# response = llm(
#     [
#         SystemMessage(
#             content=f"""
#             You are a helpful assistant.
#             If a user requests that {add_tool.description}, delegate the task to the appropriate specialized assistant by invoking the corresponding tool.
#             """
#         ),
#         user_message
#     ]
# )

# pprint(response.content)
# 사용자가 요청한 내용을 도구와 연동하여 처리
# if "add" in response.content.lower():
#     # 간단한 파싱을 통해 숫자 추출 (예제에서 하드코딩된 입력)
#     result = add_tool.run(5, 7)
#     print(f"The result of adding 5 and 7 is: {result}")
# else:
#     print("No valid tool call detected.")

KeyError: 'add_tool'

In [ ]:
from langchain.agents import initialize_agent
from langchain.chains.conversation.memory import ConversationBufferWindowMemory

# initialize conversational memory
conversational_memory = ConversationBufferWindowMemory(
        memory_key='chat_history',
        k=5,
        return_messages=True
)

agent = initialize_agent(
    # agent="chat-conversational-react-description",
    agent="conversational-description",
    tools=[add_tool],
    llm=llm,
    verbose=True,
    max_iterations=3,
    early_stopping_method='generate',
    memory=conversational_memory,
    # config={"extra": "allow"}
)

RuntimeError: no validator found for <class 'langchain.chains.llm.LLMChain'>, see `arbitrary_types_allowed` in Config

## Build Graph Design

### Constants

In [ ]:
MAIN_ASSISTANT = "main_assistant"

SPECIALIZED_TOOLS = [
    "add_tool",
]

LEAVE_TOOL_NODE = "leave_tool"

### Roles

In [ ]:
class AssistantInfo(BaseModel):
    assistant_name: str = "the four fundamental arithmetic operations calculator"

In [ ]:
assistant_info = AssistantInfo()

### Prompts

In [ ]:
PROMPT_SPECIALIZED_ASSISTANT_ENTRY_NODE = f"""
    The assistant is now the {{assistant_name}}. Reflect on the above conversation between the host assistant and the user.
    The user's intent is unsatisfied. Use the provided tools to assist the user.
    Remember, you are {{assistant_name}}, and do the four fundamental arithmetic operations, other other action is not complete until after you have successfully invoked the appropriate tool.
    If the user changes their mind or needs help for other tasks, call the FallbackToMainAssistant function to let the primary host assistant take control."
    Do not mention who you are - just act as the proxy for the assistant.",
""".format(assistant_name=assistant_info.assistant_name).strip()

### State definition

In [ ]:
# node types: state node, action node

def update_dialog_stack(left: list[str], right: Optional[str]) -> list[str]:
    """Push or pop the state."""
    if right is None:
        return left
    if right == "pop":  # stack action node name, not real node
        return left[:-1]
    return left + [right]


class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    # user_info: str
    node_trajectory: Annotated[
        list[
            Literal[
                "main_assistant",
                "add_tool",
                # "update_flight",
                # "book_car_rental",
                # "book_hotel",
                # "book_excursion",
            ]
        ],
        update_dialog_stack,
    ]

### tools

In [ ]:
@tool
def greeting():
    """
    Greeting to the users for causality tools.

    Returns:
        welcome message
    """
    greeting_message = """
    Welcome to the causality service for microplastics!
    """
    return greeting_message

In [ ]:
# Fallback Tool to main assistant
class FallbackToMainAssistant(BaseModel):
    """A tool to mark the current task as completed and/or to escalate control of the dialog to the main assistant,
    who can re-route the dialog based on the user's needs."""

    cancel: bool = True
    reason: str

    class Config:
        json_schema_extra = {
            "example": {
                "cancel": True,
                "reason": "User changed their mind about the current task.",
            },
            "example 2": {
                "cancel": True,
                "reason": "I have fully completed the task.",
            },
            "example 3": {
                "cancel": False,

                # ??
                "reason": "I need to search the user's emails or calendar for more information.",
            },
        }

In [ ]:
class FallbackToMain(BaseTool):
    name: str = ""
    description: str = """
    A tool to mark the current task as completed and/or to escalate control of the dialog to the main assistant,
    who can re-route the dialog based on the user's needs.
    """

    def _run(self, query: str):
        pass

    async def _arun(self, query: str):
        pass

In [ ]:
class AddNumbersTool(BaseTool):
    name: str = "add_numbers"
    description: str = """
    두 숫자가 주어졌을 때 숫자의 합을 계산하고 반환한다.
    """
    def _run(self, a, b):
        return str(a + b)

    async def _arun(self, a, b):
        return str(a + b)

add_tool = AddNumbersTool()